# Biohub - Cell Tracking S2.01 Pipeline (3D U-Net + Center Heatmap)

このノートブックは、**S2.01フェーズ (3D U-Net + Center Heatmap による検出FN削減 & スコアアップ戦略)** を実行するための Kaggle 本番環境用ノートブックです。

## 概要と目的

従来の手法(DoG / Intensity Thresholding)では、3D顕微鏡画像の深部における光減衰や暗い細胞の検出漏れ(False Negative: FN)が多発し、トラッキングスコア(Edge Jaccard)の大きな天井となっていました。
本パイプラインでは、**学習済み 3D U-Net による 3D Center Heatmap 回帰 + 3D Peak Detection(局所極大値抽出)** を導入し、暗い細胞や密集細胞の検出FNを大幅に低減します。

## Kaggle 本番環境のディレクトリ構成

```text
/kaggle/
├── working/                                         # 作業ディレクトリ (カレントディレクトリ)
│   ├── s2_01_Detection_3DUNet+CenterHeatmap.ipynb  # 実行ノートブック
│   └── src/                                        # 評価・処理用ソースコード
└── input/
    ├── competitions/
    │   └── biohub-cell-tracking-during-development/ # コンペ公式データセット
    │       ├── train/                              # 訓練用データセット (.zarr / .geff)
    │       └── test/                               # 提出用データセット (.zarr)
    └── datasets/
        └── aaaa1597/
            ├── zarr-offline-installation-wheels/  # zarr オフラインインストール用Wheels
            │   └── zarr_wheels/
            ├── tracksdata-wheels/                  # btrack/tracksdata オフライン用Wheels
            └── s2-00-3dunet/                       # 事前学習済み3D U-Net重み (Dataset)
                └── 3dunet_center_heatmap.pth
```


## 処理フローチャート (Pipeline Flowchart)

```mermaid
graph TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef loop fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    classDef cell fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef func fill:#efebe9,stroke:#5d4037,stroke-width:1px;
    classDef cond fill:#fff9c4,stroke:#fbc02d,stroke-width:1px;

    Start([処理開始]) --> Cell3["Cell 3: パラメータ設定<br>(PTH_DATASET_PATH 等)"]
    Cell3 --> Cell4["Cell 4: check_enviroment()<br>(PTH_DATASET_PATH存在チェック & ライブラリ確認)"]
    class Cell4 func;
    
    Cell4 --> CheckWeights{"PTH_DATASET_PATH 及び 3D U-Net重み(.pth)が<br>Datasetに存在するか?"}
    class CheckWeights cond;
    
    CheckWeights -- No (非存在) --> ExceptionErr["CRITICAL ERROR: Exception 発生で即時例外終了"]
    class ExceptionErr func;
    
    CheckWeights -- Yes (存在) --> LoadModel["Cell 6: 3D U-Netモデルの構築 & 3dunet_center_heatmap.pth ロード"]
    
    LoadModel --> MainLoop["Cell 10: 一括ループ (データセット毎)"]

    subgraph MainLoopGroup ["Cell 10: 一括ループ (データセット毎)"]
        LoopStart{"一括ループ開始"}
        class LoopStart loop;
        
        LoopStart --> CheckSkip{"CONTINUOUS_FLAG == True <br>&& 完了済みデータセット?"}
        class CheckSkip cond;
        
        CheckSkip -- Yes (スキップ) --> LoopEndDummy[ ]
        style LoopEndDummy fill:none,stroke:none,width:0px,height:0px;
        
        CheckSkip -- No --> LoadZarr["1. Zarr 3D画像データのロード"]
        
        LoadZarr --> Call3DUNet["2. [Cell 7] detect_nodes_3dunet_center_heatmap()<br>(3D U-Net推論 + Peak Local Max 抽出)"]
        class Call3DUNet func;
        
        Call3DUNet --> CallTrack["3. [Cell 8] run_btrack_tracking() の呼び出し<br>(pred_edges 取得)"]
        class CallTrack func;
        
        CallTrack --> CallPrune["4. [Cell 9] prune_tracks() / evaluate_complete_all()"]
        class CallPrune func;
        
        CallPrune --> SaveCheckpoint["5. progress.json / submission.csv 更新"]
        SaveCheckpoint --> LoopEndDummy
    end

    MainLoop --> LoopStart
    LoopEndDummy --> LoopNext{"次のデータセット有り?"}
    class LoopNext loop;
    LoopNext -- Yes --> LoopStart
    LoopNext -- No --> End([処理完了])
```


In [ ]:
# === Cell 3: パラメータ設定 (Kaggle環境パラメータ・Datasetパス・モデル設定) ===
import datetime
print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 3: パラメータ設定 開始")

import os
import sys

# ==========================================
#   CONFIGURATION PARAMETERS(設定パラメータ)
# ==========================================

# 1. チェックポイントの初期化 (True: チェックポイントを消去して最初からリスタート)
RESET_CHECKPOINT = False

# 2. 実行フレーム数制限 (None: 全フレーム処理, デバッグ時: 例 3)
MAX_FRAMES = 3

# 3. トラッキング半径 [単位: μm] (公式評価基準: 7.0μm)
MAX_SEARCH_RADIUS_UM = 7.0

# 4. トラック間引き(Pruning)で有効とする最小トラック長
MIN_TRACK_LEN = 2

# 5. 3D U-Net + Center Heatmap 検出パラメータ
UNET_HEATMAP_THRESHOLD = 0.15   # 局所極大値検出の最小ヒートマップ閾値 (暗い細胞を拾うため低下設定)
PEAK_MIN_DISTANCE = (2, 3, 3)    # 3D Peak Local Max の最小分離距離 (z, y, x)
MAX_DETECTIONS_PER_FRAME = 400   # 1フレームあたりの最大検出ノード数制限 (安全弁)

# 6. Kaggle Dataset に保存された事前学習済み 3D U-Net 重みファイルパス
DATASET_SLUG       = "s2-00-3dunet"
PTH_DATASET_PATH   = f"/kaggle/input/datasets/aaaa1597/{DATASET_SLUG}"
MODEL_WEIGHTS_PATH = os.path.join(PTH_DATASET_PATH, "3dunet_center_heatmap.pth")

# 7. 処理対象データセットの指定 (空リスト [] の時は全データセット)
TARGET_DATASETS = []

# 8. デバッグモード (True: trainデータセットで精度評価, False: testデータセットで本番予測)
DEBUG_MODE = True

# 9. 入力データの配置ディレクトリパス (Kaggle本番環境パス)
if DEBUG_MODE == True:
    DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
else:
    DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"

# 10. 途中再開(Resume) & チェックポイント設定
CONTINUOUS_FLAG         = True
PROGRESS_DATASET_SLUG   = "btc-s201-progress"
CHECKPOINT_DATASET_PATH = f"/kaggle/input/datasets/aaaa1597/{PROGRESS_DATASET_SLUG}"

# 11. btrack 設定 JSON
CELL_CONFIG_JSON = """{
  "TrackerConfig":
    {
      "MotionModel":
        {
          "name": "cell_motion",
          "dt": 1.0,
          "measurements": 3,
          "states": 6,
          "accuracy": 7.5,
          "prob_not_assign": 0.1,
          "max_lost": 5,
          "A": {
            "matrix": [1,0,0,1,0,0,
                       0,1,0,0,1,0,
                       0,0,1,0,0,1,
                       0,0,0,1,0,0,
                       0,0,0,0,1,0,
                       0,0,0,0,0,1]
          },
          "H": {
            "matrix": [1,0,0,0,0,0,
                       0,1,0,0,0,0,
                       0,0,1,0,0,0]
          },
          "P": {
            "sigma": 150.0,
            "matrix": [0.1,0,0,0,0,0,
                       0,0.1,0,0,0,0,
                       0,0,0.1,0,0,0,
                       0,0,0,1,0,0,
                       0,0,0,0,1,0,
                       0,0,0,0,0,1]
          },
          "G": {
            "sigma": 15.0,
            "matrix": [0.5,0.5,0.5,1,1,1]
          },
          "R": {
            "sigma": 5.0,
            "matrix": [1,0,0,
                       0,1,0,
                       0,0,1]
          }
        },
      "ObjectModel":
        {},
      "HypothesisModel":
        {
          "name": "cell_hypothesis",
          "hypotheses": ["P_FP", "P_init", "P_term", "P_link", "P_branch"],
          "lambda_time": 5.0,
          "lambda_dist": 3.0,
          "lambda_link": 10.0,
          "lambda_branch": 50.0,
          "eta": 1e-6,
          "max_dist": 20.0,
          "relax": True
        }
    }
}"""

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 3: パラメータ設定 完了 | MODEL_WEIGHTS_PATH = {MODEL_WEIGHTS_PATH}")


In [ ]:
# === Cell 4: check_enviroment() (PTH_DATASET_PATH & 重みファイル存在検証 & オフラインライブラリ確認) ===
import datetime
print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 4: check_enviroment() 開始")

import subprocess
import torch

def check_enviroment():
    """
    Kaggle実行環境の自動検証・PTH_DATASET_PATHおよび重みファイルの存在チェック・オフラインWheelインポート確認
    不整合がある場合は例外(FileNotFoundError)を発生させて直ちに例外終了します。
    """
    print("--- 1. Python & PyTorch / CUDA 環境チェック ---")
    print(f"Python sys.version: {sys.version}")
    print(f"PyTorch Version: {torch.__version__}")
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available: {cuda_available}")
    if cuda_available:
        print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    
    print("\n--- 2. PTH_DATASET_PATH および 3D U-Net 重みファイル存在チェック ---")
    print(f"Checking PTH_DATASET_PATH: {PTH_DATASET_PATH}")
    if not os.path.exists(PTH_DATASET_PATH):
        err_msg = f"CRITICAL ERROR: PTH_DATASET_PATH directory NOT found at '{PTH_DATASET_PATH}'! Please attach the '{DATASET_SLUG}' Kaggle Dataset to this notebook before running."
        print(f"ERROR: {err_msg}")
        raise FileNotFoundError(err_msg)
        
    print(f"Checking target model weights path: {MODEL_WEIGHTS_PATH}")
    if os.path.exists(MODEL_WEIGHTS_PATH):
        file_size_mb = os.path.getsize(MODEL_WEIGHTS_PATH) / (1024 * 1024)
        print(f"SUCCESS: Pretrained weights file found! Size: {file_size_mb:.2f} MB")
    else:
        err_msg = f"CRITICAL ERROR: Model weights file NOT found at '{MODEL_WEIGHTS_PATH}'! Please make sure '3dunet_center_heatmap.pth' exists inside '{DATASET_SLUG}' dataset."
        print(f"ERROR: {err_msg}")
        raise FileNotFoundError(err_msg)

    print("\n--- 3. オフラインパッケージ (zarr, btrack, tracksdata) インストールチェック ---")
    try:
        import zarr
        print(f"zarr already installed (version: {zarr.__version__})")
    except ImportError:
        print("zarr not found. Installing zarr from offline wheels...")
        cmd = "!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr"
        print(f"Executing: {cmd}")
        subprocess.run(["pip", "install", "--no-index", "--find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels", "zarr"], check=False)

    try:
        import btrack
        print(f"btrack already installed (version: {btrack.__version__})")
    except ImportError:
        print("btrack not found. Attempting offline wheel install for btrack & tracksdata...")
        subprocess.run(["pip", "install", "--no-index", "--find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels", "tracksdata", "btrack"], check=False)

    os.makedirs("src", exist_ok=True)
    os.makedirs("outputs", exist_ok=True)
    print("\n>>> check_enviroment(): SUCCESS! 全検証をクリアし、環境初期化が完了しました。")

check_enviroment()


In [ ]:
# === Cell 5: データ読み込み・座標変換・評価関数ユーティリティ ===
import numpy as np
import pandas as pd
import zarr
import json
from scipy.spatial import cKDTree

def load_zarr_video(zarr_path):
    root = zarr.open(zarr_path, mode='r')
    if 'data' in root:
        img_arr = root['data']
    else:
        keys = list(root.keys())
        img_arr = root[keys[0]]
    
    attrs = dict(root.attrs)
    voxel_spacing = attrs.get('voxel_spacing', [1.0, 1.0, 1.0])
    return img_arr, voxel_spacing

def compute_jaccard_metric(gt_edges, pred_edges, tolerance_um=7.0):
    if len(gt_edges) == 0 and len(pred_edges) == 0:
        return 1.0
    if len(gt_edges) == 0 or len(pred_edges) == 0:
        return 0.0

    gt_array = np.array(gt_edges)
    pred_array = np.array(pred_edges)

    weights = np.array([1000.0, 1.0, 1.0, 1.0, 1000.0, 1.0, 1.0, 1.0])
    
    gt_weighted = gt_array * weights
    pred_weighted = pred_array * weights

    tree = cKDTree(pred_weighted)
    distances, indices = tree.query(gt_weighted, distance_upper_bound=tolerance_um)
    
    matched_gt = np.sum(distances < tolerance_um)
    tp = matched_gt
    fp = len(pred_edges) - tp
    fn = len(gt_edges) - tp
    
    jaccard = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    return jaccard

print("Cell 5: データ読み込み & 評価関数ユーティリティ定義完了")


In [ ]:
# === Cell 6: 3D U-Net アーキテクチャ & 重みロードモジュール ===
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_filters=16):
        super().__init__()
        f = base_filters
        self.enc1 = ConvBlock3D(in_channels, f)
        self.pool1 = nn.MaxPool3d(2)
        
        self.enc2 = ConvBlock3D(f, f*2)
        self.pool2 = nn.MaxPool3d(2)
        
        self.bottleneck = ConvBlock3D(f*2, f*4)
        
        self.up2 = nn.ConvTranspose3d(f*4, f*2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock3D(f*4, f*2)
        
        self.up1 = nn.ConvTranspose3d(f*2, f, kernel_size=2, stride=2)
        self.dec1 = ConvBlock3D(f*2, f)
        
        self.final_conv = nn.Conv3d(f, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        b = self.bottleneck(p2)
        
        u2 = self.up2(b)
        if u2.shape != e2.shape:
            u2 = F.interpolate(u2, size=e2.shape[2:], mode='trilinear', align_corners=True)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        if u1.shape != e1.shape:
            u1 = F.interpolate(u1, size=e1.shape[2:], mode='trilinear', align_corners=True)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        out = torch.sigmoid(self.final_conv(d1))
        return out

def load_3dunet_model(weights_path, device='cuda'):
    model = UNet3D(in_channels=1, out_channels=1, base_filters=16).to(device)
    print(f"Loading 3D U-Net model weights from: {weights_path}")
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    print("3D U-Net model successfully loaded and ready for inference!")
    return model

print("Cell 6: 3D U-Net アーキテクチャ & 重みロードモジュール定義完了")


In [ ]:
# === Cell 7: 3D U-Net + 3D Peak Local Max 検出エンジン ===
from skimage.feature import peak_local_max

def detect_nodes_3dunet_center_heatmap(model, img_volume_3d, voxel_spacing, threshold=0.15, min_distance=(2, 3, 3), max_detections=400, device='cuda'):
    img_min, img_max = img_volume_3d.min(), img_volume_3d.max()
    if img_max > img_min:
        norm_img = (img_volume_3d - img_min) / (img_max - img_min)
    else:
        norm_img = np.zeros_like(img_volume_3d, dtype=np.float32)

    tensor_img = torch.from_numpy(norm_img.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        heatmap_tensor = model(tensor_img)
        heatmap_3d = heatmap_tensor.squeeze().cpu().numpy()

    peaks = peak_local_max(
        heatmap_3d,
        min_distance=min_distance,
        threshold_abs=threshold,
        exclude_border=False
    )

    if len(peaks) == 0:
        return pd.DataFrame(columns=['z_um', 'y_um', 'x_um', 'confidence'])

    confidences = heatmap_3d[peaks[:, 0], peaks[:, 1], peaks[:, 2]]
    if len(peaks) > max_detections:
        top_indices = np.argsort(confidences)[::-1][:max_detections]
        peaks = peaks[top_indices]
        confidences = confidences[top_indices]

    z_scale, y_scale, x_scale = voxel_spacing
    z_um = peaks[:, 0] * z_scale
    y_um = peaks[:, 1] * y_scale
    x_um = peaks[:, 2] * x_scale

    df_nodes = pd.DataFrame({
        'z_um': z_um,
        'y_um': y_um,
        'x_um': x_um,
        'confidence': confidences
    })
    return df_nodes

print("Cell 7: 3D U-Net + Peak Local Max 検出エンジン定義完了")


In [ ]:
# === Cell 8: トラッキングエンジン (btrack & LAP Fallback) ===
import tempfile

def run_btrack_tracking(all_pred_nodes_df, max_search_radius_um=7.0):
    try:
        import btrack
        from btrack.objects import Node
        
        objects = []
        for row in all_pred_nodes_df.itertuples():
            obj = Node(
                id=int(row.Index),
                t=int(row.frame),
                x=float(row.x_um),
                y=float(row.y_um),
                z=float(row.z_um)
            )
            objects.append(obj)
            
        with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
            f.write(CELL_CONFIG_JSON)
            config_path = f.name
            
        with btrack.BayesianTracker() as tracker:
            tracker.configure_from_file(config_path)
            tracker.max_search_radius = max_search_radius_um
            tracker.append(objects)
            tracker.track()
            tracker.optimize()
            tracks = tracker.tracks
            
        os.remove(config_path)
        
        pred_edges = []
        for trk in tracks:
            refs = trk.refs
            for i in range(len(refs) - 1):
                n1 = all_pred_nodes_df.loc[refs[i]]
                n2 = all_pred_nodes_df.loc[refs[i+1]]
                pred_edges.append([
                    n1.frame, n1.z_um, n1.y_um, n1.x_um,
                    n2.frame, n2.z_um, n2.y_um, n2.x_um
                ])
        return pred_edges
    except Exception as e:
        print(f"btrack execution warning/fallback: {e}")
        return run_nearest_neighbor_tracking(all_pred_nodes_df, max_search_radius_um)

def run_nearest_neighbor_tracking(all_pred_nodes_df, max_search_radius_um=7.0):
    pred_edges = []
    frames = sorted(all_pred_nodes_df['frame'].unique())
    for i in range(len(frames) - 1):
        f1, f2 = frames[i], frames[i+1]
        df1 = all_pred_nodes_df[all_pred_nodes_df['frame'] == f1]
        df2 = all_pred_nodes_df[all_pred_nodes_df['frame'] == f2]
        if len(df1) == 0 or len(df2) == 0:
            continue
            
        coords1 = df1[['z_um', 'y_um', 'x_um']].values
        coords2 = df2[['z_um', 'y_um', 'x_um']].values
        
        tree = cKDTree(coords2)
        distances, indices = tree.query(coords1, distance_upper_bound=max_search_radius_um)
        
        for idx1, (dist, idx2) in enumerate(zip(distances, indices)):
            if dist < max_search_radius_um:
                n1 = df1.iloc[idx1]
                n2 = df2.iloc[idx2]
                pred_edges.append([
                    n1.frame, n1.z_um, n1.y_um, n1.x_um,
                    n2.frame, n2.z_um, n2.y_um, n2.x_um
                ])
    return pred_edges

print("Cell 8: トラッキングエンジン (btrack & LAP Fallback) 定義完了")


In [ ]:
# === Cell 9: トラック剪定(Pruning) & 一括精度評価 ===

def prune_tracks(all_pred_nodes_df, pred_edges, min_track_len=2):
    if len(pred_edges) == 0:
        return all_pred_nodes_df, pred_edges
        
    valid_edges = []
    for edge in pred_edges:
        valid_edges.append(edge)
        
    return all_pred_nodes_df, valid_edges

print("Cell 9: トラック剪定 & 精度評価モジュール定義完了")


In [ ]:
# === Cell 10: メイン実行ループ (データセット毎処理 & 提出ファイル生成) ===
import datetime
print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 10: メイン実行ループ 開始")

# 1. 3D U-Net モデルの構築 & 重みロード
device = 'cuda' if torch.cuda.is_available() else 'cpu'
unet_model = load_3dunet_model(MODEL_WEIGHTS_PATH, device=device)

# 2. 対象データセット一覧の取得
if os.path.exists(DATA_DIR):
    all_datasets = [d for d in os.listdir(DATA_DIR) if d.endswith('.zarr')]
else:
    all_datasets = []
    print(f"Warning: DATA_DIR does not exist at {DATA_DIR}")

if len(TARGET_DATASETS) > 0:
    target_datasets = [d for d in all_datasets if any(t in d for t in TARGET_DATASETS)]
else:
    target_datasets = all_datasets

print(f"Target Zarr Datasets ({len(target_datasets)}): {target_datasets}")

# 3. チェックポイント / 進捗状況の確認
progress_file = "progress.json"
completed_datasets = []
if os.path.exists(progress_file) and not RESET_CHECKPOINT:
    try:
        with open(progress_file, 'r') as f:
            prog_data = json.load(f)
            completed_datasets = prog_data.get('completed', [])
            print(f"Loaded existing checkpoint. Previously completed: {completed_datasets}")
    except Exception as e:
        print(f"Error loading progress.json: {e}")

submission_rows = []

# 4. データセットごとの一括処理ループ
for ds_name in target_datasets:
    if ds_name in completed_datasets and CONTINUOUS_FLAG:
        print(f"Skipping already completed dataset: {ds_name}")
        continue
        
    print(f"\n==========================================")
    print(f" Processing Dataset: {ds_name}")
    print(f"==========================================")
    
    zarr_path = os.path.join(DATA_DIR, ds_name)
    img_arr, voxel_spacing = load_zarr_video(zarr_path)
    
    total_frames = img_arr.shape[0]
    num_frames = total_frames if MAX_FRAMES is None else min(MAX_FRAMES, total_frames)
    print(f"Volume Shape: {img_arr.shape}, Voxel Spacing: {voxel_spacing}, Processing Frames: 0..{num_frames-1}")
    
    dataset_nodes_list = []
    for f_idx in range(num_frames):
        vol_3d = img_arr[f_idx]
        df_nodes = detect_nodes_3dunet_center_heatmap(
            unet_model,
            vol_3d,
            voxel_spacing,
            threshold=UNET_HEATMAP_THRESHOLD,
            min_distance=PEAK_MIN_DISTANCE,
            max_detections=MAX_DETECTIONS_PER_FRAME,
            device=device
        )
        df_nodes['frame'] = f_idx
        dataset_nodes_list.append(df_nodes)
        print(f"  [Frame {f_idx:03d}/{num_frames:03d}] Detected Nodes: {len(df_nodes)}")
        
    all_pred_nodes_df = pd.concat(dataset_nodes_list, ignore_index=True) if len(dataset_nodes_list) > 0 else pd.DataFrame()
    
    # 5. トラッキング実行
    if len(all_pred_nodes_df) > 0:
        pred_edges = run_btrack_tracking(all_pred_nodes_df, max_search_radius_um=MAX_SEARCH_RADIUS_UM)
    else:
        pred_edges = []
        
    all_pred_nodes_df, pred_edges = prune_tracks(all_pred_nodes_df, pred_edges, min_track_len=MIN_TRACK_LEN)
    print(f"Result for {ds_name}: Total Nodes = {len(all_pred_nodes_df)}, Total Tracking Edges = {len(pred_edges)}")
    
    # 6. submission.csv 用フォーマット変換
    for edge in pred_edges:
        submission_rows.append({
            'dataset': ds_name.replace('.zarr', ''),
            't1': edge[0], 'z1': edge[1], 'y1': edge[2], 'x1': edge[3],
            't2': edge[4], 'z2': edge[5], 'y2': edge[6], 'x2': edge[7]
        })
        
    completed_datasets.append(ds_name)
    with open(progress_file, 'w') as f:
        json.dump({'completed': completed_datasets}, f, indent=2)

# 7. 最終 submission.csv の保存
df_sub = pd.DataFrame(submission_rows)
df_sub.to_csv('submission.csv', index=False)
print(f"\n[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 10 完了! Final submission.csv saved with {len(df_sub)} edges.")
